In [ ]:
import random
from IPython.display import Image
from pgmpy.utils import get_example_model

model_name = "child"
# model_name = "alarm"
# model_name = "insurance"
seed = 0
random.seed(seed)

model_original = get_example_model(model_name)
print(model_original.nodes())
for (i, j) in model_original.edges():
    print(f"{i} -> {j}")

model_original.to_graphviz().draw("asia_graph.png", prog="dot")

In [ ]:
# Prepare data
import openbnsllib
from helpers.pgmpy_bridge import to_pgmpy, to_openbnsl
from helpers.structural_distance import structural_errors, PDAG2CPDAG

samples = model_original.simulate(int(1e3), seed=seed)
samples = samples[sorted(samples.columns)]

df_wrapper = openbnsllib.base.DataframeWrapper(samples)
# citest_type = openbnsllib.citest.ChiSquare(level=0.05)
oracle_graph = to_openbnsl(model_original, df_wrapper.col_str2idx)
citest_type = openbnsllib.citest.OracleGraph(oracle_graph)

print(df_wrapper.col_str2idx)


In [ ]:
# Run PC
pdag_learned_by_pc = openbnsllib.structure_learning.pc(
    df_wrapper, 
    citest_type, 
    max_cond_vars=len(samples.columns),
)
pgmpy_pdag_learned_by_pc = to_pgmpy(pdag_learned_by_pc, list(samples.columns))
pgmpy_pdag_learned_by_pc = PDAG2CPDAG(pgmpy_pdag_learned_by_pc)

In [ ]:
# Run RAI
pdag_learned_by_rai = openbnsllib.structure_learning.rai(
    df_wrapper, 
    citest_type, 
    max_cond_vars=len(samples.columns),
)
pgmpy_pdag_learned_by_rai = to_pgmpy(pdag_learned_by_rai, list(samples.columns))
pgmpy_pdag_learned_by_rai = PDAG2CPDAG(pgmpy_pdag_learned_by_rai)


In [ ]:
print(f"Structural errors for {model_name} (PC): {structural_errors(model_original, pgmpy_pdag_learned_by_pc)}")
print(f"Structural errors for {model_name} (RAI): {structural_errors(model_original, pgmpy_pdag_learned_by_rai)}")

In [ ]:
import pygraphviz as pgv

def draw_pdag(pdag, filename):
    edges = list(pdag.edges())
    directed_edges = []
    undirected_edges = set()

    for u, v in edges:
        if (v, u) in edges:
            undirected_edges.add(tuple(sorted((u, v))))
        else:
            directed_edges.append((u, v))

    lines = ["digraph G {"]
    for u, v in directed_edges:
        lines.append(f'    "{u}" -> "{v}";')
    for u, v in undirected_edges:
        lines.append(f'    "{u}" -> "{v}" [dir=none];')
    lines.append("}")
    dot_str = "\n".join(lines)

    A = pgv.AGraph(string=dot_str)
    A.draw(filename, prog="dot")


draw_pdag(pgmpy_pdag_learned_by_pc,  "pgmpy_pdag_learned_by_pc.png")
draw_pdag(pgmpy_pdag_learned_by_rai, "pgmpy_pdag_learned_by_rai.png")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img1 = mpimg.imread("asia_graph.png")
img2 = mpimg.imread("pgmpy_pdag_learned_by_pc.png")
img3 = mpimg.imread("pgmpy_pdag_learned_by_rai.png")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img1)
axes[0].set_title("Original Model")
axes[0].axis("off")

axes[1].imshow(img2)
axes[1].set_title("Learned Model (PC)")
axes[1].axis("off")

axes[2].imshow(img3)
axes[2].set_title("Learned Model (RAI)")
axes[2].axis("off")

plt.tight_layout()
plt.show()